# Train Maddie — Apple Silicon (M5) Local

Train Maddie **locally** on M5 Apple Silicon using **MLX** — leverages GPU Neural Accelerators and unified memory. No Colab, no cloud.

- **MLX** uses Metal 4 + TensorOps → M5 Neural Accelerators (macOS 26.2+)
- **LoRA** fine-tuning for memory efficiency
- **Presets**: wikitext, tiny_stories, alpaca, code_expert — or custom URL

In [1]:
!echo $VIRTUAL_ENV

/Users/koushikbabu/Developer/LLM/venv


## 1. Apple Silicon setup (M5 / M4 / M3 / M2)

Detect Apple Silicon and verify MLX will use GPU + Neural Accelerators.

In [5]:
import platform
import subprocess

# Apple Silicon detection
IS_APPLE_SILICON = platform.machine() == "arm64" and platform.system() == "Darwin"
if not IS_APPLE_SILICON:
    print("WARNING: Not on Apple Silicon. MLX works best on M1/M2/M3/M4/M5.")
else:
    # Get chip info (M1, M2, M3, M4, M5, etc.)
    try:
        result = subprocess.run(["sysctl", "-n", "machdep.cpu.brand_string"], capture_output=True, text=True)
        chip = result.stdout.strip() if result.returncode == 0 else "Apple Silicon"
        print(f"Running on: {chip}")
        if "M5" in chip or "M4" in chip:
            print("Neural Accelerators available (MLX uses Metal 4 TensorOps)")
    except Exception:
        print("Apple Silicon detected")

# Local output paths (no Drive)
OUTPUT_BASE = "./maddie_mlx"
ADAPTER_PATH = f"{OUTPUT_BASE}/adapters"
DATA_DIR = f"{OUTPUT_BASE}/data"

Running on: Apple M5
Neural Accelerators available (MLX uses Metal 4 TensorOps)


## 2. Install MLX and dependencies

In [6]:
# MLX and mlx-lm with training support
# MLX uses Apple's Metal → GPU + Neural Accelerators (M5)
# subprocess avoids zsh globbing [] in mlx-lm[train]
import subprocess
subprocess.run(["pip", "install", "-q", "mlx", "mlx-lm[train]", "datasets"], check=True)


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


CompletedProcess(args=['pip', 'install', '-q', 'mlx', 'mlx-lm[train]', 'datasets'], returncode=0)

### Dataset options (pick one)

| Preset | What it is | Best for | Size |
|--------|------------|----------|------|
| **wikitext** | Wikipedia articles (raw text) | General language modeling | ~2M tokens |
| **tiny_stories** | Short simple stories | Story-style generation, small LM | ~2M tokens |
| **alpaca** | Instruction (instruction + response) | Maddie as a helpful Q&A bot | ~52k examples |
| **code_expert** | Code instructions + solutions | Maddie as a **coding expert** | ~20k examples |
| **custom** | Your own file | Any format you have a **direct download URL** for | — |

- Use **preset** = one of `wikitext`, `tiny_stories`, `alpaca`, `code_expert` — no URL needed.
- Use **custom** = set `DATASET_URL` below and we'll download it via terminal.

In [7]:
# Config: base model (MLX supports Llama, Mistral, Qwen2, Phi2, etc. — not GPT-2)
# Use mlx-community for pre-converted models, or HF repo (mlx auto-converts)
BASE_MODEL = "mlx-community/Qwen2-1.5B-Instruct-4bit"  # 4-bit fits ~4GB; use Qwen2-1.5B-Instruct for full precision
MAX_ITERS = 500      # Training iterations (adjust for your time)
BATCH_SIZE = 2       # Reduce to 1 if OOM on smaller Macs

### Dataset options (pick one)

| Preset | What it is | Best for | Size |
|--------|------------|----------|------|
| **wikitext** | Wikipedia articles (raw text) | General language modeling | ~2M tokens |
| **tiny_stories** | Short simple stories | Story-style generation, small LM | ~2M tokens |
| **alpaca** | Instruction (instruction + response) | Maddie as a helpful Q&A bot | ~52k examples |
| **code_expert** | Code instructions + solutions | Maddie as a **coding expert** | ~20k examples |
| **custom** | Your own file | Any format you have a **direct download URL** for | — |

- Use **preset** = one of `wikitext`, `tiny_stories`, `alpaca`, `code_expert` — no URL needed.
- Use **custom** = set `DATASET_URL` below and we’ll download it via terminal.

## 3. Get dataset

In [8]:
import os
import json

USE_PRESET = True
PRESET = "wikitext"  # wikitext | tiny_stories | alpaca | code_expert
DATASET_URL = "https://example.com/your_dataset.jsonl"  # when USE_PRESET=False

os.makedirs(DATA_DIR, exist_ok=True)

if USE_PRESET:
    from datasets import load_dataset
    if PRESET == "wikitext":
        data = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
        text_column = "text"
    elif PRESET == "tiny_stories":
        data = load_dataset("roneneldan/TinyStories", split="train")
        data = data.select(range(min(50000, len(data))))
        text_column = "text"
    elif PRESET == "alpaca":
        data = load_dataset("tatsu-lab/alpaca", split="train")
        def alpaca_to_text(ex):
            inp = ex.get("input") or ""
            return f"Instruction: {ex['instruction']}\nInput: {inp}\nResponse: {ex['output']}"
        data = data.map(lambda ex: {"text": alpaca_to_text(ex)}, remove_columns=data.column_names)
        text_column = "text"
    elif PRESET == "code_expert":
        data = load_dataset("HuggingFaceH4/CodeAlpaca_20k", split="train")
        def code_to_text(ex):
            return f"Instruction: {ex['prompt']}\nResponse: {ex['completion']}"
        data = data.map(lambda ex: {"text": code_to_text(ex)}, remove_columns=data.column_names)
        text_column = "text"
    else:
        raise ValueError(f"Unknown PRESET: {PRESET}. Use wikitext, tiny_stories, alpaca, or code_expert.")
    # Write train.jsonl for mlx-lm (filter empty lines)
    train_path = os.path.join(DATA_DIR, "train.jsonl")
    with open(train_path, "w") as f:
        for row in data:
            t = row.get(text_column, "").strip()
            if t:
                f.write(json.dumps({"text": t}) + "\n")
    print(f"Loaded preset '{PRESET}': {len(data)} rows → {train_path}")
else:
    filename = os.path.basename(DATASET_URL.split("?")[0]) or "dataset.jsonl"
    filepath = os.path.join(DATA_DIR, filename)
    if not os.path.exists(filepath) or os.path.getsize(filepath) == 0:
        get_ipython().system(f'curl -L -o "{filepath}" "{DATASET_URL}"')
    # Copy to train.jsonl if custom format matches
    train_path = os.path.join(DATA_DIR, "train.jsonl")
    with open(filepath) as src, open(train_path, "w") as dst:
        for line in src:
            line = line.strip()
            if line:
                try:
                    obj = json.loads(line)
                    dst.write(json.dumps({"text": obj.get("text", str(obj))}) + "\n")
                except json.JSONDecodeError:
                    dst.write(json.dumps({"text": line}) + "\n")
    print(f"Dataset at: {train_path}")

/Users/koushikbabu/Developer/LLM/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded preset 'wikitext': 36718 rows → ./maddie_mlx/data/train.jsonl


In [9]:
# Optional: create valid.jsonl for validation loss (e.g. 10% of train)
# mlx-lm will use it if present in DATA_DIR


## 4. Train Maddie with MLX LoRA

In [10]:
# Train with mlx_lm.lora — runs on Apple Silicon GPU + Neural Accelerators
import subprocess

cmd = [
    "python", "-m", "mlx_lm.lora",
    "--model", BASE_MODEL,
    "--train",
    "--data", DATA_DIR,
    "--adapter-path", ADAPTER_PATH,
    "--iters", str(MAX_ITERS),
    "--batch-size", str(BATCH_SIZE),
    "--learning-rate", "1e-5",
    "--save-every", "100",
    "--steps-per-report", "10",
    "--max-seq-length", "512",
]
# Gradient checkpointing saves memory on smaller Macs
# cmd.extend(["--grad-checkpoint"])
subprocess.run(cmd)

Calling `python -m mlx_lm.lora...` directly is deprecated. Use `mlx_lm.lora...` or `python -m mlx_lm lora ...` instead.
Loading pretrained model


Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 23937.06it/s]


Loading datasets
Training
Trainable parameters: 0.342% (5.276M/1543.714M)
Starting training..., iters: 500


mx.metal.device_info is deprecated and will be removed in a future version. Use mx.device_info instead.


Iter 10: Train loss 3.387, Learning Rate 1.000e-05, It/sec 2.535, Tokens/sec 405.370, Trained Tokens 1599, Peak mem 3.108 GB
Iter 20: Train loss 3.098, Learning Rate 1.000e-05, It/sec 3.358, Tokens/sec 452.973, Trained Tokens 2948, Peak mem 3.108 GB
Iter 30: Train loss 3.129, Learning Rate 1.000e-05, It/sec 2.854, Tokens/sec 555.145, Trained Tokens 4893, Peak mem 3.108 GB
Iter 40: Train loss 3.192, Learning Rate 1.000e-05, It/sec 2.612, Tokens/sec 505.610, Trained Tokens 6829, Peak mem 4.047 GB
Iter 50: Train loss 2.699, Learning Rate 1.000e-05, It/sec 2.258, Tokens/sec 533.410, Trained Tokens 9191, Peak mem 4.047 GB
Iter 60: Train loss 2.713, Learning Rate 1.000e-05, It/sec 2.624, Tokens/sec 431.327, Trained Tokens 10835, Peak mem 4.047 GB
Iter 70: Train loss 3.310, Learning Rate 1.000e-05, It/sec 3.236, Tokens/sec 461.759, Trained Tokens 12262, Peak mem 4.047 GB
Iter 80: Train loss 2.998, Learning Rate 1.000e-05, It/sec 2.567, Tokens/sec 584.263, Trained Tokens 14538, Peak mem 4.047 

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/koushikbabu/Developer/LLM/venv/lib/python3.11/site-packages/mlx_lm/lora.py", line 370, in <module>
    main()
  File "/Users/koushikbabu/Developer/LLM/venv/lib/python3.11/site-packages/mlx_lm/lora.py", line 362, in main
    run(types.SimpleNamespace(**args))
  File "/Users/koushikbabu/Developer/LLM/venv/lib/python3.11/site-packages/mlx_lm/lora.py", line 334, in run
    train_model(args, model, train_set, valid_set, training_callback)
  File "/Users/koushikbabu/Developer/LLM/venv/lib/python3.11/site-packages/mlx_lm/lora.py", line 288, in train_model
    train(
  File "/Users/koushikbabu/Developer/LLM/venv/lib/python3.11/site-packages/mlx_lm/tuner/trainer.py", line 314, in train
    mx.eval(state, losses, n_tokens, grad_accum)
KeyboardInterrupt


KeyboardInterrupt: 

In [4]:
# Adapters saved to ADAPTER_PATH. Optional: fuse adapters into full model
# get_ipython().system(f'python -m mlx_lm.fuse --model {BASE_MODEL} --adapter-path {ADAPTER_PATH} --save-path {OUTPUT_BASE}/fused')
print(f"Maddie adapters saved to {ADAPTER_PATH}")

NameError: name 'ADAPTER_PATH' is not defined

## 5. Save Maddie (adapters)

In [11]:
# LoRA adapters are saved automatically during training to ADAPTER_PATH
# To fuse adapters into a single model (optional):
# get_ipython().system(f'python -m mlx_lm.fuse --model {BASE_MODEL} --adapter-path {ADAPTER_PATH} --save-path {OUTPUT_BASE}/Maddie_fused')
print(f"Adapters: {ADAPTER_PATH}")

Adapters: ./maddie_mlx/adapters


## 6. Load & Use Trained Maddie

In [3]:
# Generate with Maddie (MLX uses Apple Silicon GPU + Neural Accelerators)
import subprocess

prompt = "Instruction: Write a Python function to reverse a string\nResponse:"
result = subprocess.run(
    ["python", "-m", "mlx_lm.generate",
     "--model", BASE_MODEL,
     "--adapter-path", ADAPTER_PATH,
     "--prompt", prompt,
     "--max-tokens", "200",
     "--temp", "0.3"],
    capture_output=True, text=True
)
print("="*60)
print("PROMPT:")
print(prompt)
print("="*60)
print("MADDIE'S RESPONSE:")
print(result.stdout.split("Response:")[-1].strip() if result.returncode == 0 else result.stderr)

NameError: name 'ADAPTER_PATH' is not defined

### Try different prompts (MLX)

In [13]:
def maddie_generate(prompt, max_tokens=200, temp=0.3):
    """Generate with Maddie using MLX (Apple Silicon)."""
    r = subprocess.run(
        ["python", "-m", "mlx_lm.generate",
         "--model", BASE_MODEL, "--adapter-path", ADAPTER_PATH,
         "--prompt", prompt, "--max-tokens", str(max_tokens), "--temp", str(temp)],
        capture_output=True, text=True
    )
    return r.stdout.split("Response:")[-1].strip() if r.returncode == 0 else r.stderr

# Change this prompt to test Maddie
prompt = "Instruction: Write a function to calculate factorial\nResponse:"
print("Maddie:", maddie_generate(prompt))

Maddie: Calling `python -m mlx_lm.generate...` directly is deprecated. Use `mlx_lm.generate...` or `python -m mlx_lm generate ...` instead.

No text generated for this prompt


### Try different prompts

In [19]:
# Chat loop with Maddie (MLX)
print("="*60)
print("Chat with Maddie! Type 'quit' or 'exit' to stop.")
print("="*60 + "\n")

while True:
    user_input = input("You: ").strip()
    if user_input.lower() in ['quit', 'exit', 'q']:
        print("\nGoodbye! 👋")
        break
    if not user_input:
        continue
    prompt = f"Instruction: {user_input}\nResponse:"
    print(f"\nMaddie: {maddie_generate(prompt)}\n")

Chat with Maddie! Type 'quit' or 'exit' to stop.


Goodbye! 👋


### Or continue from a prompt

In [15]:
# Continue from a starting text
starting_text = "Machine learning is a subset of artificial intelligence"
print(f"Starting: {starting_text}\n")
print(f"Maddie continues:\n{maddie_generate(starting_text, max_tokens=150, temp=0.7)}")

Starting: Machine learning is a subset of artificial intelligence

Maddie continues:
Calling `python -m mlx_lm.generate...` directly is deprecated. Use `mlx_lm.generate...` or `python -m mlx_lm generate ...` instead.
AI refers to any technology or system that simulates or emulates the human mind or intelligence , including : Problem solving , emotion , perception , learning , reasoning , speech , and language .
Prompt: 27 tokens, 172.683 tokens-per-sec
Generation: 38 tokens, 93.148 tokens-per-sec
Peak memory: 0.964 GB


### Quick test (no chat loop)

In [16]:
# One-off generation
starting_text = "Machine learning is a subset of artificial intelligence"
print(f"Starting: {starting_text}\n")
print(f"Maddie continues:\n{maddie_generate(starting_text, max_tokens=150, temp=0.7)}")

Starting: Machine learning is a subset of artificial intelligence

Maddie continues:
Calling `python -m mlx_lm.generate...` directly is deprecated. Use `mlx_lm.generate...` or `python -m mlx_lm generate ...` instead.
Yes, that is correct. Machine learning is a subset of artificial intelligence that involves using algorithms and statistical models to enable computers to learn and make predictions based on patterns in data.
Prompt: 27 tokens, 389.870 tokens-per-sec
Generation: 36 tokens, 95.968 tokens-per-sec
Peak memory: 0.964 GB
